# 🏴󠁧󠁢󠁥󠁮󠁧󠁿 Premier League Match Predictor — Exploration des données

Ce notebook analyse les données scrapées depuis **Understat.com** (saison 2023-2024) et entraîne le modèle de Machine Learning.

**Pipeline complet :**
1. Chargement et aperçu des données
2. Statistiques descriptives
3. Visualisations (résultats, xG, équipes)
4. Entraînement du modèle Random Forest
5. Évaluation et prédiction personnalisée

> **Prérequis :** avoir lancé `uv run python scraper/scrape_understat.py` pour générer `data/matches.csv`

## 1. Chargement des données

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('../data/matches.csv')

print(f'Matchs chargés  : {len(df)}')
print(f'Colonnes        : {list(df.columns)}')
print(f'Période         : {df["date"].min()} → {df["date"].max()}')
print(f'Équipes         : {df["home"].nunique()}')
df.head(8)

## 2. Statistiques descriptives

In [ ]:
print('=== Valeurs manquantes ===')
print(df.isnull().sum())

print('\n=== Statistiques numériques ===')
df[['score_home', 'score_away', 'xg_home', 'xg_away']].describe().round(2)

## 3. Distribution des résultats

In [ ]:
labels_map = {'H': 'Victoire\ndomicile', 'D': 'Match nul', 'A': 'Victoire\nextérieur'}
colors     = ['#1a78cf', '#aaaaaa', '#d62728']
counts     = df['result'].value_counts().reindex(['H', 'D', 'A'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barplot
axes[0].bar([labels_map[k] for k in counts.index], counts.values, color=colors, edgecolor='white', linewidth=1.2)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Nombre de matchs par résultat', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nombre de matchs')
axes[0].set_ylim(0, counts.max() + 25)

# Camembert
axes[1].pie(
    counts.values,
    labels=[labels_map[k] for k in counts.index],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Répartition des résultats (%)', fontsize=13, fontweight='bold')

plt.suptitle('Distribution des résultats — Premier League 2023-2024', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/distribution_resultats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé dans data/distribution_resultats.png')

## 4. Analyse des Expected Goals (xG)

Le **xG (Expected Goals)** mesure la qualité des occasions créées.
Un xG de 2.0 signifie qu'une équipe a créé l'équivalent de 2 buts attendus — indépendamment du score final.

In [ ]:
df_clean = df.dropna(subset=['xg_home', 'xg_away']).copy()
df_clean['xg_diff'] = df_clean['xg_home'] - df_clean['xg_away']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution des xG
axes[0].hist(df_clean['xg_home'], bins=25, alpha=0.7, label='xG Domicile', color='#1a78cf')
axes[0].hist(df_clean['xg_away'], bins=25, alpha=0.7, label='xG Extérieur', color='#d62728')
axes[0].axvline(df_clean['xg_home'].mean(), color='#1a78cf', linestyle='--', label=f'Moy. dom. {df_clean["xg_home"].mean():.2f}')
axes[0].axvline(df_clean['xg_away'].mean(), color='#d62728', linestyle='--', label=f'Moy. ext. {df_clean["xg_away"].mean():.2f}')
axes[0].set_title('Distribution des xG')
axes[0].set_xlabel('Expected Goals')
axes[0].legend(fontsize=8)

# xG diff par résultat
palette = {'H': '#1a78cf', 'D': '#aaaaaa', 'A': '#d62728'}
sns.boxplot(data=df_clean, x='result', y='xg_diff', order=['H', 'D', 'A'],
            palette=palette, ax=axes[1])
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Différence de xG selon le résultat')
axes[1].set_xlabel('Résultat  (H = domicile, D = nul, A = extérieur)')
axes[1].set_ylabel('xG domicile − xG extérieur')

# xG home vs away scatter coloré par résultat
for res, color in palette.items():
    subset = df_clean[df_clean['result'] == res]
    axes[2].scatter(subset['xg_home'], subset['xg_away'], alpha=0.4, s=20, color=color,
                    label={'H': 'Victoire dom.', 'D': 'Nul', 'A': 'Victoire ext.'}[res])
axes[2].plot([0, 5], [0, 5], 'k--', linewidth=0.8, label='Égalité xG')
axes[2].set_title('xG domicile vs extérieur')
axes[2].set_xlabel('xG domicile')
axes[2].set_ylabel('xG extérieur')
axes[2].legend(fontsize=8)

plt.suptitle('Analyse des Expected Goals (xG) — Understat.com', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/analyse_xg.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Top équipes par xG moyen

In [ ]:
# xG moyen à domicile et à l'extérieur par équipe
xg_home_avg = df.groupby('home')['xg_home'].mean().rename('xG domicile')
xg_away_avg = df.groupby('away')['xg_away'].mean().rename('xG extérieur')
xg_team     = pd.concat([xg_home_avg, xg_away_avg], axis=1)
xg_team['xG moyen'] = xg_team.mean(axis=1)
xg_team = xg_team.sort_values('xG moyen', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(xg_team.index, xg_team['xG moyen'], color='#1a78cf', edgecolor='white')
ax.axvline(xg_team['xG moyen'].mean(), color='red', linestyle='--', label='Moyenne ligue')
ax.set_title('xG moyen par équipe — Premier League 2023-2024', fontsize=13, fontweight='bold')
ax.set_xlabel('Expected Goals moyen par match')
ax.legend()
plt.tight_layout()
plt.savefig('../data/xg_par_equipe.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Entraînement du modèle Random Forest

In [ ]:
import sys
sys.path.append('..')
from ml.predict import load_and_prepare, train_model

X, y = load_and_prepare('../data/matches.csv')
print(f'Features : {list(X.columns)}')
print(f'Taille du dataset : {len(X)} matchs')
print(f'Distribution des classes :\n{y.value_counts()}')

In [ ]:
model = train_model(X, y)
print('\nModèle entraîné avec succès !')
print('→ Voir data/confusion_matrix.png et data/feature_importance.png')

## 7. Prédiction personnalisée

In [ ]:
from ml.predict import predict_match

# ── Modifiez ces valeurs pour simuler un match ──
xg_home = 2.1   # xG estimé pour l'équipe à domicile
xg_away = 0.9   # xG estimé pour l'équipe à l'extérieur

print('Simulation : équipe domicile (xG=2.1) vs équipe extérieur (xG=0.9)')
predict_match(model, xg_home=xg_home, xg_away=xg_away)

---
## Conclusion

Ce notebook démontre le pipeline complet du projet :

1. **Scraping** : données xG réelles récupérées sur Understat.com via Playwright
2. **Exploration** : distribution des résultats (~45% victoire domicile, attendu en football), corrélation entre xG et résultat
3. **Machine Learning** : Random Forest atteignant ~58–65% de précision
4. **Application** : prédictions accessibles via `uv run python app/app.py`

> La précision de 58–65% est cohérente avec la littérature : le football reste un sport où la chance joue un rôle important. L'objectif principal est d'illustrer le pipeline complet de la donnée à la prédiction.